# einops-repeat-broadcast — faded example 2: Pair every point with every cluster center

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `einops-repeat-broadcast`. Running the beacon reports progress on the `Einops: Repeat-as-broadcast` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Repeat-as-broadcast` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`einops-repeat-broadcast`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-repeat-broadcast"
DD_SUBTOPIC = "Einops: Repeat-as-broadcast"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The k-means assignment step needs every point compared against every center. `einops.repeat` builds the `(N, K, D)` pair grid as two stride-zero views, one expanding points over centers and one expanding centers over points, so the squared-distance reduction over `D` gives an `(N, K)` matrix without copies.

## Faded exercise 2

Implement `point_center_dists(points, centers)`. `points` is `(N, D)`, `centers` is `(K, D)`. Build both `(N, K, D)` broadcast views with `einops.repeat`, then return the `(N, K)` matrix of squared distances (sum of squared differences over `D`). Complete the blanked broadcast of `centers` over the point axis.

**Fill in:** the einops.repeat that expands centers to (N, K, D) by inserting the leading point axis

In [ ]:
import torch as t
import einops
from einops import repeat

t.manual_seed(4)
points = t.randn(6, 3)
centers = t.randn(4, 3)

def point_center_dists(points, centers):
    N = points.shape[0]
    K = centers.shape[0]
    p = repeat(points, 'n d -> n k d', k=K)
    c = repeat(centers, 'k d -> n k d', n=N)
    return ((p - c) ** 2).sum(dim=-1)

print(point_center_dists(points, centers).shape)


def _test():
    out = point_center_dists(points, centers)
    assert out.shape == (6, 4), out.shape
    # independent ground truth via torch.cdist (squared)
    gt = t.cdist(points, centers) ** 2
    assert t.allclose(out, gt, atol=1e-4), (out - gt).abs().max()


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import einops
from einops import repeat

t.manual_seed(4)
points = t.randn(6, 3)
centers = t.randn(4, 3)

def point_center_dists(points, centers):
    N = points.shape[0]
    K = centers.shape[0]
    p = repeat(points, 'n d -> n k d', k=K)
    c = repeat(centers, 'k d -> n k d', n=N)
    return ((p - c) ** 2).sum(dim=-1)

print(point_center_dists(points, centers).shape)
```
</details>